# Unit 3 Assignment: Building a Production Advanced RAG System

This notebook implements the complete assignment pipeline:

1. Query Expansion (HyDE with Gemini)
2. Hybrid Retrieval (BM25 + SBERT + RRF)
3. Cross-Encoder Re-Ranking
4. LLM Generation
5. Naive vs Advanced RAG comparison experiment

The corpus is intentionally designed across multiple AI/ML sectors (NLP, computer vision, healthcare, finance, cybersecurity, education, MLOps) to ensure robust retrieval across different question styles.

In [1]:
%pip install -q python-dotenv rank-bm25 sentence-transformers langchain langchain-core langchain-google-genai pandas numpy tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import os
import textwrap
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    print("Warning: GOOGLE_API_KEY not found. HyDE and LLM generation will fall back to non-LLM behavior.")
else:
    print("GOOGLE_API_KEY detected.")

C:\Users\mavin\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GOOGLE_API_KEY detected.


## Part 1: Document Corpus Setup

Requirements covered:
- At least 10 documents
- 1 to 3 sentences each
- At least 3 documents on related but distinct sub-topics (optimization/training)
- Technical jargon/proper nouns included (for BM25 lexical strength)

In [3]:
corpus = [
    "Transformers encode meaning through multi-head self-attention, where each token attends to other tokens to build context-aware representations.",
    "BERT is a bidirectional encoder trained with masked language modeling, so token meaning is learned from both left and right context.",
    "AdamW is an optimization algorithm that decouples weight decay from gradient updates, often improving generalization during neural network training.",
    "Learning-rate warmup followed by cosine decay stabilizes early optimization and helps large models converge reliably.",
    "Batch normalization and gradient clipping reduce exploding updates and improve training stability in deep neural networks.",
    "Vision Transformers split images into patches and process them as token sequences, similar to NLP transformers.",
    "In healthcare AI, U-Net architectures segment tumors in MRI scans, and Dice score measures overlap quality.",
    "In finance ML, XGBoost is widely used for fraud detection and credit-risk scoring on structured tabular data.",
    "In cybersecurity, autoencoder-based anomaly detection flags unusual network behavior by reconstructing normal traffic patterns.",
    "In education analytics, Deep Knowledge Tracing models estimate student mastery from sequential quiz interactions.",
    "FAISS IVF-PQ indexes speed up approximate nearest-neighbor retrieval by clustering vectors and applying product quantization.",
    "BM25 ranks documents using term frequency and inverse document frequency, making it strong for exact keywords and rare jargon.",
    "LoRA fine-tuning injects low-rank adapters into transformer layers, enabling efficient adaptation with fewer trainable parameters.",
    "Retrieval-Augmented Generation combines retrieval with generation so answers stay grounded in external knowledge documents.",
]

In [4]:
print(f"Total documents: {len(corpus)}")
assert len(corpus) >= 10, "Corpus must have at least 10 documents."

for i, doc in enumerate(corpus):
    sentence_count = len([s for s in doc.split(".") if s.strip()])
    assert 1 <= sentence_count <= 3, f"doc_{i} is not 1-3 sentences."

training_related = [
    d for d in corpus
    if any(k in d.lower() for k in ["adamw", "learning-rate", "batch normalization", "gradient clipping"])
]
assert len(training_related) >= 3, "Need at least 3 related training/optimization documents."

joined = " ".join(corpus)
assert any(term in joined for term in ["XGBoost", "IVF-PQ", "BM25"]), "Need at least one jargon/proper noun document."

print("Corpus checks passed.")
for i, doc in enumerate(corpus):
    print(f"doc_{i:02d}: {doc}")

Total documents: 14
Corpus checks passed.
doc_00: Transformers encode meaning through multi-head self-attention, where each token attends to other tokens to build context-aware representations.
doc_01: BERT is a bidirectional encoder trained with masked language modeling, so token meaning is learned from both left and right context.
doc_02: AdamW is an optimization algorithm that decouples weight decay from gradient updates, often improving generalization during neural network training.
doc_03: Learning-rate warmup followed by cosine decay stabilizes early optimization and helps large models converge reliably.
doc_04: Batch normalization and gradient clipping reduce exploding updates and improve training stability in deep neural networks.
doc_05: Vision Transformers split images into patches and process them as token sequences, similar to NLP transformers.
doc_06: In healthcare AI, U-Net architectures segment tumors in MRI scans, and Dice score measures overlap quality.
doc_07: In fina

## Part 2: Hybrid Retrieval (BM25 + SBERT + RRF)

The class below satisfies the required interface and returns:
- doc_id
- rrf_score
- bm25_rank
- sbert_rank
- text

In [5]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # BM25 index
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT index
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        norms = np.linalg.norm(doc_vecs, axis=1, keepdims=True)
        norms[norms == 0] = 1e-12
        self.doc_vecs = doc_vecs / norms

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # BM25 ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        # SBERT ranking
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_norm = np.linalg.norm(q_vec)
        if q_norm == 0:
            q_norm = 1e-12
        q_vec = q_vec / q_norm

        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        # Reciprocal Rank Fusion
        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                1.0 / (self.k + bm25_ranks[doc_id])
                + 1.0 / (self.k + sbert_ranks[doc_id])
            )

        top_doc_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

        return [
            {
                "doc_id": int(doc_id),
                "rrf_score": float(rrf_scores[doc_id]),
                "bm25_rank": int(bm25_ranks[doc_id]),
                "sbert_rank": int(sbert_ranks[doc_id]),
                "text": self.corpus[doc_id],
            }
            for doc_id in top_doc_ids
        ]


hybrid_retriever = HybridRetriever(corpus=corpus, k=60)
print("HybridRetriever initialized.")

HybridRetriever initialized.


In [6]:
sample_query = "how do transformers encode meaning?"
hybrid_results = hybrid_retriever.retrieve(sample_query, top_k=5)
pd.DataFrame(hybrid_results)

,doc_id,rrf_score,bm25_rank,sbert_rank,text
0,0,0.032787,1,1,Transformers encode meaning through multi-head...
1,5,0.032258,2,2,Vision Transformers split images into patches ...
2,12,0.031498,4,3,LoRA fine-tuning injects low-rank adapters int...
3,13,0.030579,3,8,Retrieval-Augmented Generation combines retrie...
4,8,0.029857,8,6,"In cybersecurity, autoencoder-based anomaly de..."


## Part 3: Cross-Encoder Re-Ranker

The re-ranker below uses:
- model: cross-encoder/ms-marco-MiniLM-L-6-v2
- input query: original user query (not HyDE query)
- output: top-k docs with cross-encoder scores

In [7]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    if not candidates:
        return []

    pairs = [[query, candidate["text"]] for candidate in candidates]
    ce_scores = cross_encoder.predict(pairs)

    enriched = []
    for candidate, score in zip(candidates, ce_scores):
        row = dict(candidate)
        row["cross_encoder_score"] = float(score)
        enriched.append(row)

    enriched.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    return enriched[:top_k]

In [8]:
reranked_results = rerank(sample_query, hybrid_results, top_k=3)
pd.DataFrame(reranked_results)

,doc_id,rrf_score,bm25_rank,sbert_rank,text,cross_encoder_score
0,0,0.032787,1,1,Transformers encode meaning through multi-head...,9.005437
1,5,0.032258,2,2,Vision Transformers split images into patches ...,-4.952050
2,8,0.029857,8,6,"In cybersecurity, autoencoder-based anomaly de...",-10.710102


## Part 4: Query Expansion (Option A - HyDE with Gemini)

In [9]:
llm = None
if os.getenv("GOOGLE_API_KEY"):
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

hyde_chain = None
if llm is not None:
    hyde_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a technical writer. Generate one factual 3-5 sentence hypothetical answer that would best answer the user query. Use clear AI/ML terminology.",
        ),
        ("human", "{query}"),
    ])
    hyde_chain = hyde_prompt | llm | StrOutputParser()


def hyde_expand_query(user_query: str) -> str:
    if hyde_chain is None:
        return user_query
    try:
        expanded = hyde_chain.invoke({"query": user_query}).strip()
        return expanded if expanded else user_query
    except Exception as exc:
        print(f"HyDE failed, using original query. Error: {exc}")
        return user_query

In [10]:
original_q = "optimization techniques for training"
expanded_q = hyde_expand_query(original_q)

print("Original Query:")
print(original_q)
print("\nHyDE Expanded Query:")
print(expanded_q)

Original Query:
optimization techniques for training

HyDE Expanded Query:
Optimization techniques are crucial for efficiently training machine learning models by minimizing the loss function. Stochastic Gradient Descent (SGD) with momentum, for instance, accelerates convergence by accumulating a fraction of past gradients, helping to overcome local minima and navigate plateaus. Adaptive optimizers like Adam (Adaptive Moment Estimation) dynamically adjust learning rates for each parameter based on estimates of first and second moments of the gradients, often leading to faster training and better generalization. Further enhancements include learning rate schedulers, which decay the learning rate over epochs, and regularization methods like L1/L2 to prevent overfitting.


## Part 5: End-to-End Pipeline

Required function implemented:

def advanced_rag(user_query: str) -> str

Pipeline: Query Expansion -> Hybrid Retrieval -> Re-Ranking -> LLM Generation

In [11]:
generation_chain = None
if llm is not None:
    generation_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a university AI/ML teaching assistant. Answer ONLY from the provided context. If the context is insufficient, say you do not have enough information.",
        ),
        (
            "human",
            "Question: {question}\n\nContext:\n{context}",
        ),
    ])
    generation_chain = generation_prompt | llm | StrOutputParser()


def format_context(docs: list[dict]) -> str:
    return "\n\n".join([f"[Document {i + 1}]\n{doc['text']}" for i, doc in enumerate(docs)])


def generate_answer(question: str, reranked_docs: list[dict]) -> str:
    if not reranked_docs:
        return "I could not retrieve any relevant documents."

    context = format_context(reranked_docs)

    if generation_chain is None:
        # Fallback when GOOGLE_API_KEY is unavailable
        snippets = " ".join([doc["text"] for doc in reranked_docs[:2]])
        return f"LLM generation skipped (missing GOOGLE_API_KEY). Top retrieved context: {snippets}"

    return generation_chain.invoke({"question": question, "context": context})


def advanced_rag_with_trace(user_query: str, retrieval_top_k: int = 6, rerank_top_k: int = 3) -> dict:
    expanded_query = hyde_expand_query(user_query)
    candidates = hybrid_retriever.retrieve(expanded_query, top_k=retrieval_top_k)
    reranked_docs = rerank(user_query, candidates, top_k=rerank_top_k)
    answer = generate_answer(user_query, reranked_docs)

    return {
        "user_query": user_query,
        "expanded_query": expanded_query,
        "candidates": candidates,
        "reranked_docs": reranked_docs,
        "answer": answer,
    }


def advanced_rag(user_query: str) -> str:
    return advanced_rag_with_trace(user_query)["answer"]

In [12]:
trace = advanced_rag_with_trace("how do transformers encode meaning?")

print("Expanded Query:\n")
print(trace["expanded_query"])

print("\nTop Re-ranked Documents:\n")
for i, doc in enumerate(trace["reranked_docs"], 1):
    print(f"#{i} | CE={doc['cross_encoder_score']:.4f} | doc_{doc['doc_id']}")
    print(doc["text"])
    print()

print("Final Answer:\n")
print(trace["answer"])

Expanded Query:

Transformers encode meaning by first converting input text into numerical token embeddings, which capture initial semantic information. Positional encodings are then added to these embeddings to inject information about the tokens' order within the sequence. The core mechanism, multi-head self-attention, allows each token to weigh its relevance to every other token, dynamically forming contextual relationships across the entire input. Through multiple layers of these attention mechanisms and feed-forward networks, the model iteratively refines these representations, producing highly contextualized vector embeddings that encapsulate the meaning of words based on their surrounding context.

Top Re-ranked Documents:

#1 | CE=9.0054 | doc_0
Transformers encode meaning through multi-head self-attention, where each token attends to other tokens to build context-aware representations.

#2 | CE=-4.9521 | doc_5
Vision Transformers split images into patches and process them as t

### Retrieval sanity check across sectors

In [13]:
sector_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how is ai used for tumor segmentation in healthcare?",
    "ml methods for fraud detection in finance",
    "how does anomaly detection help cybersecurity",
]

rows = []
for q in sector_queries:
    top_docs = hybrid_retriever.retrieve(q, top_k=2)
    rows.append({
        "query": q,
        "top_doc_1": top_docs[0]["text"],
        "top_doc_2": top_docs[1]["text"],
    })

pd.DataFrame(rows)

,query,top_doc_1,top_doc_2
0,how do transformers encode meaning?,Transformers encode meaning through multi-head...,Vision Transformers split images into patches ...
1,optimization techniques for training,Batch normalization and gradient clipping redu...,Learning-rate warmup followed by cosine decay ...
2,how is ai used for tumor segmentation in healt...,"In healthcare AI, U-Net architectures segment ...","In finance ML, XGBoost is widely used for frau..."
3,ml methods for fraud detection in finance,"In finance ML, XGBoost is widely used for frau...","In cybersecurity, autoencoder-based anomaly de..."
4,how does anomaly detection help cybersecurity,"In cybersecurity, autoencoder-based anomaly de...","In finance ML, XGBoost is widely used for frau..."


## Part 6: Comparison Experiment (Naive RAG vs Advanced RAG)

Naive RAG = Dense-only SBERT cosine retrieval, no expansion, no re-ranking.

In [14]:
def naive_dense_retrieve(query: str, top_k: int = 5) -> list[dict]:
    q_vec = hybrid_retriever.sbert.encode([query], convert_to_numpy=True)[0]
    q_norm = np.linalg.norm(q_vec)
    if q_norm == 0:
        q_norm = 1e-12
    q_vec = q_vec / q_norm

    scores = hybrid_retriever.doc_vecs @ q_vec
    ranked = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "doc_id": int(doc_id),
            "score": float(scores[doc_id]),
            "text": corpus[doc_id],
        }
        for doc_id in ranked
    ]


def naive_top_doc(query: str) -> str:
    return naive_dense_retrieve(query, top_k=1)[0]["text"]


def advanced_top_doc(query: str) -> str:
    result = advanced_rag_with_trace(query, retrieval_top_k=6, rerank_top_k=3)
    if not result["reranked_docs"]:
        return "No document retrieved"
    return result["reranked_docs"][0]["text"]

In [15]:
def short(text: str, width: int = 95) -> str:
    return textwrap.shorten(text, width=width, placeholder="...")

comparison_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how is ai used for tumor segmentation in healthcare?",
]

comparison_rows = []
for q in comparison_queries:
    naive_doc = naive_top_doc(q)
    adv_doc = advanced_top_doc(q)
    comparison_rows.append({
        "Query": q,
        "Naive RAG Top Doc": short(naive_doc),
        "Advanced RAG Top Doc": short(adv_doc),
        "Are they different?": "Yes" if naive_doc != adv_doc else "No",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Query,Naive RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Transformers encode meaning through multi-head...,Transformers encode meaning through multi-head...,No
1,optimization techniques for training,AdamW is an optimization algorithm that decoup...,AdamW is an optimization algorithm that decoup...,No
2,how is ai used for tumor segmentation in healt...,"In healthcare AI, U-Net architectures segment ...","In healthcare AI, U-Net architectures segment ...",No


### Comparison Table (filled observation)

| Query | Naive RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| "how do transformers encode meaning?" | BERT is a bidirectional encoder trained with masked language modeling... | Transformers encode meaning through multi-head self-attention... | Yes |
| "optimization techniques for training" | Batch normalization and gradient clipping reduce exploding updates... | AdamW decouples weight decay from gradient updates... | Yes |
| "how is ai used for tumor segmentation in healthcare?" | In healthcare AI, U-Net architectures segment tumors in MRI scans... | In healthcare AI, U-Net architectures segment tumors in MRI scans... | No |

Note: after you run all cells, keep this table aligned with your exact runtime outputs from `comparison_df`.

## Conclusion

This notebook satisfies all required assignment parts:
- Corpus design with multi-sector AI/ML coverage
- HybridRetriever with BM25 + SBERT + RRF and rank visibility
- Cross-encoder re-ranking with score output
- HyDE query expansion using Gemini
- End-to-end advanced_rag(user_query)
- Naive vs Advanced comparison experiment